# Pretraining of the U-Net of `munich_new_site`

**Purpose.** This notebook pretrains the U-Net on the simulated scene variants of `maps.npz`.

**Inputs.** `lab/data/munich_new_site/raw/maps.npz` and the files of `lab/data/munich_new_site/processed/`.

**Method.** The method follows Geo2SigMap (arXiv:2312.14303). A path gain map of the Sionna RT
`RadioMapSolver` on the original scene replaces its U-Net-Iso. For cell $c$ and grid cell
$g$, the U-Net $f_\theta$ gives the path gain in dB directly,

$$\hat{G}_{c}(g) = \mu + \sigma \, f_\theta\big(\mathbf{x}_c\big)(g) \quad \text{(dB)},$$

where $\mathbf{x}_c$ has the input channels of Table 2, and $\mu$ and $\sigma$ are the mean and
the standard deviation of the input path gain in dB (`Normalization`). The simulated path gain
is an input channel, not a base map. A grid cell with no ray gets the floor
`Normalization.floor_db` in this channel. The output layer starts at zero, thus a new model
gives $\mu$ everywhere. The ablation of `munich_ablation` compares this output with a residual
output $G^{\mathrm{RT}}_{c}(g) + f_\theta(\mathbf{x}_c)(g)$. The target of the pretraining is the
path gain $G^{\star}_{c}$ of the same cells on a changed scene. The loss is the mean
absolute error (MAE) in dB, over the grid cells $\mathcal{G}$ with a target more than 0,

$$\mathcal{L} = \frac{1}{|\mathcal{G}|} \sum_{g \in \mathcal{G}}
\left| \max\big(G^{\star}_c(g), G_{\mathrm{floor}}\big) - \hat{G}_c(g) \right| \quad \text{(dB)},$$

where $G_{\mathrm{floor}}$ is `Normalization.floor_db`. A loss in dB has a gradient also for a
large error. The SMAPE of the linear values is almost constant for an error of more than about
20 dB, and with it the pretraining stopped on a plateau for some seeds. The U-Net has 3 levels with GroupNorm and
bilinear up-sampling: `radio_calibration/calibration/unet.py` gives the sources. The settings
are in `lab/configs/calibration.yaml`.

**Outputs.** `lab/outputs/munich_new_site/models/pretrained*.pt` and `lab/outputs/munich_new_site/histories/`.

**Requirements.** Run `task scenario -- scenario=munich_new_site` and `02_preprocessing.ipynb` first.
The notebook uses the GPU if one is available.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from experiments import history_figure, load_scenario, pretrain_model, save_model
from radio_calibration.evaluation import ResourceMonitor
from utils import load_config, save_figure, save_table, setup_notebook

cfg = load_config("scenario=munich_new_site")
device = setup_notebook(cfg)
data = load_scenario(cfg)

## 2. Data and model

Table 1 gives the pretraining data. Each layout is one sample for each of its cells. All cells
of one layout go to the same part. Table 2 gives the input channels.

In [ ]:
from radio_calibration.calibration import FEATURE_NAMES, make_network
from utils import calibration_settings

calibration = calibration_settings(cfg)
maps = data.maps
num_validation = max(1, round(calibration.validation_fraction * len(maps.variants)))
display(
    save_table(
        pd.Series(
            {
                "Layouts": len(maps.variants),
                "Cells in a layout": maps.variants[0].num_cells,
                "Samples (layout, cell)": sum(v.num_cells for v in maps.variants),
                "Validation layouts": num_validation,
                "Grid rows and columns": f"{maps.target.grid.ny} x {maps.target.grid.nx}",
            }
        )
        .astype(str)
        .to_frame("Value")
        .style.set_caption("Table 1. Pretraining data."),
        "pretraining_data",
    )
)
descriptions = {
    "path_gain_db": "Sionna RT path gain in dB with the floor, normalized",
    "has_path": "1 where a ray reaches the grid cell",
    "max_height": "Highest building height in the grid cell, normalized",
    "building_fraction": "Fraction of the grid cell with a building",
    "los": "1 where the antenna has a line of sight (geometric ray)",
    "distance": "Horizontal distance to the antenna / diagonal of the grid",
}
network = make_network(len(FEATURE_NAMES), calibration)
save_table(
    pd.Series(descriptions)
    .rename_axis("Channel")
    .to_frame("Description")
    .style.set_caption(
        f"Table 2. Input channels. The U-Net has {sum(p.numel() for p in network.parameters()):,} "
        "parameters."
    ),
    "pretraining_channels",
)

## 3. Pretraining

In [ ]:
with ResourceMonitor() as monitor:
    model, history = pretrain_model(cfg, data, device)
save_model(data, "pretrained", model, history)
histories = {"pretrained": history}
resources = {"pretrained": monitor.summary()}

In [ ]:
fig = history_figure(histories)
save_figure(fig, "pretraining_history")
plt.show()

**Fig. 1.** Pretraining. (a) MAE of the train layouts in each epoch. (b) MAE of the
validation layouts. The kept parameters have the smallest validation MAE.

In [ ]:
save_table(
    pd.DataFrame(
        {
            model_name: {
                "Smallest validation MAE (dB)": history["validation_loss"].min(),
                "Epoch of the smallest validation MAE": history["validation_loss"].idxmin() + 1,
                "Mean time of one step (s)": history["step_seconds"].mean(),
            }
            for model_name, history in histories.items()
        }
    )
    .T.style.set_caption("Table 3. Result of the pretraining.")
    .format(precision=4),
    "pretraining_result",
)

In [ ]:
save_table(
    pd.DataFrame(resources)
    .T.style.set_caption("Table 4. Resource use of the pretraining.")
    .format(precision=1),
    "pretraining_resources",
)